# QC: preprocess, filtering, normalization, log tansform

### Documention

In [ ]:
# https://nbisweden.github.io/excelerate-scRNAseq/session-qc/Quality_control.html

## Load packages

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import scrublet as scr
import numpy as np
import matplotlib.pyplot as plt
import mygene

## Load the h5ad Data Object containing the counts and annotations

In [ ]:
combined_adata = ad.read_h5ad('/storage/users/data/PANC/H5AD_file/combined_adata.h5ad')

In [ ]:
combined_adata

In [ ]:
print(combined_adata.obs['condition'])

In [ ]:
combined_adata.var_names

In [ ]:
print(combined_adata)

## QC: preprocess, filtering, normalization

### Filter: CMO tags

In [ ]:
# Identify CMO tags (or other spike-ins)
combined_adata.var["CMO"] = combined_adata.var_names.str.contains("CMO")

In [ ]:
combined_adata.var["CMO"]

In [ ]:
# Get Percentage of CMOs in cells
# Filter the Data for CMO Tags
cmo_data = combined_adata[:, combined_adata.var["CMO"].values].X
# Calculate the Percentage:
cmo_counts_per_cell = cmo_data.sum(axis=1)
total_counts_per_cell = combined_adata.X.sum(axis=1)
pct_cmo_per_cell = (cmo_counts_per_cell / total_counts_per_cell) * 100
# Add to the obs DataFrame:
combined_adata.obs["pct_cmo"] = pct_cmo_per_cell
print(combined_adata.obs["pct_cmo"])

#### Remove CMO tags

In [ ]:
combined_adata.var["CMO"]

In [ ]:
combined_adata = combined_adata[:, ~combined_adata.var["CMO"].values]

In [ ]:
combined_adata.var["CMO"]

In [ ]:
combined_adata

### Calculate several QC metrics that will be stored in the adata Object

In [ ]:
sc.pp.calculate_qc_metrics(combined_adata, inplace=True)
print(combined_adata)

### Filter: Mitochondrial and Ribosomal Genes

#### Create dictionary ensembl2symbol

In [ ]:
import mygene

# Extract Ensembl Gene IDs from combined_adata
ensembl_ids = combined_adata.var_names.tolist()

# Initialize MyGene.info client
mg = mygene.MyGeneInfo()

# Query MyGene.info for mappings
print("Querying MyGene.info...")
results = mg.querymany(
    ensembl_ids,
    scopes="ensembl.gene",
    fields="symbol",
    species="human"
)

# Initialize an empty dictionary to store the mapping
ensembl_to_gene_name = {}

# Populate the dictionary
print("Processing results...")
for res in results:
    ensembl_id = res.get("query")
    if "notfound" in res:
        # Skip if not found
        continue
    # Use the symbol field as the gene name
    ensembl_to_gene_name[ensembl_id] = res.get("symbol", ensembl_id)

# Display the resulting dictionary
print("\nEnsembl to Gene Name mapping:")
#print(ensembl_to_gene_name)


In [ ]:
for i, (ensembl_id, gene_name) in enumerate(ensembl_to_gene_name.items()):
    if i >= 35:
        break
    print(f"{ensembl_id} : {gene_name}")

#### Add gene symbol to combined_adata object

##### Annotate Gene Symbol

In [ ]:
# Assuming ensembl_to_gene_name is a dictionary where the keys are Ensembl IDs and the values are gene symbols
combined_adata.var["gene_symbol"] = combined_adata.var_names.map(ensembl_to_gene_name)

In [ ]:
combined_adata.var["gene_symbol"]

In [ ]:
combined_adata

##### Fill empty Symbols with Ensembl IDs

In [ ]:
combined_adata.var["ensembl_gene_id"] = combined_adata.var_names.astype(str)
combined_adata.var["gene_symbol"].fillna(combined_adata.var["ensembl_gene_id"], inplace=True)
print("Remaining NaN in 'gene_symbol':", combined_adata.var["gene_symbol"].isna().sum())


In [ ]:
combined_adata.var_names

In [ ]:
combined_adata.var["gene_symbol"]

In [ ]:
combined_adata

#### Analyze Mito and Ribo Genes

In [ ]:
# After loading your data, identify mitochondrial and ribosomal genes
#combined_adata.var["mito"] = combined_adata.var_names.str.startswith("MT-") | combined_adata.var_names.str.startswith("mt-")
#combined_adata.var["ribo"] = combined_adata.var_names.str.startswith("RPS") | combined_adata.var_names.str.startswith("RPL")

# Using the gene_symbol column to identify mitochondrial and ribosomal genes
combined_adata.var["mito"] = combined_adata.var["gene_symbol"].str.startswith("MT-") | combined_adata.var["gene_symbol"].str.startswith("mt-")
combined_adata.var["ribo"] = combined_adata.var["gene_symbol"].str.startswith("RPS") | combined_adata.var["gene_symbol"].str.startswith("RPL")


In [ ]:
combined_adata.var["mito"]

#### Compute QC metrics, considering both mitochondrial and ribosomal genes

In [ ]:
# Compute QC metrics, considering both mitochondrial and ribosomal genes
sc.pp.calculate_qc_metrics(combined_adata, qc_vars=["mito", "ribo"], percent_top=None, inplace=True)

In [ ]:
# Get a list of the newly annotated adata object and the Mitochondrial count
print(combined_adata)
combined_adata.obs.pct_counts_ribo

In [ ]:
# Sort the cells by Percentage Mitochondrium and order by highest percentage first
sorted_adata = combined_adata[combined_adata.obs['pct_counts_mito'].sort_values(ascending=False).index]
sorted_adata.obs.pct_counts_mito

#### Visualize cell's mitchondrial and ribosomal gene expression

In [ ]:
# Visualize basic QC metrics, including ribosomal content:
sc.pl.violin(combined_adata, ['n_genes_by_counts', 'total_counts'])
sc.pl.violin(combined_adata, ['pct_counts_mito','pct_counts_ribo'])
import scanpy as sc

sc.pl.violin(
    combined_adata,
    keys=['pct_counts_mito', 'pct_counts_ribo'],
    groupby='condition',
    jitter=0.4,
    rotation=45,
    stripplot=True,
    multi_panel=True,
    save='violin_mito_ribo.png'  # Save the plot as an image
)
plt.show()

sc.pl.scatter(combined_adata, x='total_counts', y='pct_counts_mito')
sc.pl.scatter(combined_adata, x='total_counts', y='pct_counts_ribo')
sc.pl.scatter(combined_adata, x='total_counts', y='n_genes_by_counts')



#### Filter Mito and Ribo according to a certain percentage (e.g. 10 and 50%)

In [ ]:
combined_adata.obs['condition']

##### Not used!!: Delete faulty ribo/mito: new logic to filter conditions wise

In [ ]:
import numpy as np
import scanpy as sc

# Recalculate QC metrics if needed
sc.pp.calculate_qc_metrics(
    combined_adata,
    qc_vars=["mito", "ribo"],
    percent_top=None,
    inplace=True
)

# Initialize dicts to store 95th-percentile thresholds per condition
mito_thresholds = {}
ribo_thresholds = {}

# Compute per-condition upper cutoffs
for condition in combined_adata.obs['condition'].unique():
    # Subset obs for this condition
    cond_obs = combined_adata.obs.loc[
        combined_adata.obs['condition'] == condition
    ]

    if cond_obs.empty:
        print(f"Skipping condition: {condition} (no cells)")
        continue

    mito_up = np.percentile(cond_obs['pct_counts_mito'], 95)
    ribo_up = np.percentile(cond_obs['pct_counts_ribo'], 95)

    mito_thresholds[condition] = mito_up
    ribo_thresholds[condition] = ribo_up

    print(f"{condition}: mito ≤ {mito_up:.2f}%, ribo ≤ {ribo_up:.2f}%")

# Apply filtering: keep cells whose mito AND ribo percentages are below their condition-specific 95th percentile
filtered_adata = combined_adata[
    combined_adata.obs.apply(
        lambda x: (
            x['pct_counts_mito'] <= mito_thresholds[x['condition']] and
            x['pct_counts_ribo'] <= ribo_thresholds[x['condition']]
        ),
        axis=1
    ),
    :
]

print(f"Filtered data shape: {filtered_adata.shape}")


##### Delete faulty ribo/mito: classic easy logic to filter conditions wise

In [ ]:
filtered_adata = combined_adata.copy()

In [ ]:
# Filter cells based on mitochondrial and ribosomal content:
# Cells with high mitochondrial gene expression might be undergoing apoptosis, so you might want to exclude them.
sc.pp.calculate_qc_metrics(filtered_adata, qc_vars=["mito", "ribo"], percent_top=None, inplace=True)
filtered_adata = filtered_adata[filtered_adata.obs.pct_counts_mito < 10, :]
# high levels of ribosomal RNA (rRNA) can indicate incomplete poly-A tail capture or contamination. Thus, by filtering out cells with excessive ribosomal transcripts, we're likely removing lower-quality cells.
filtered_adata = filtered_adata[filtered_adata.obs.pct_counts_ribo < 50, :]

In [ ]:
filtered_adata

#### Redefine object according to which mito and ribo filter method used

In [ ]:
filtered_combined_adata = filtered_adata.copy()

#### View and evaluate filtering

In [ ]:
sc.pp.calculate_qc_metrics(filtered_combined_adata, qc_vars=["mito", "ribo"], percent_top=None, inplace=True)
sc.pl.violin(
    filtered_combined_adata,
    keys=['pct_counts_mito', 'pct_counts_ribo'],
    groupby='condition',
    jitter=0.4,
    rotation=45,
    stripplot=True,
    multi_panel=True,
    save='violin_mito_ribo.png'  # Save the plot as an image
)
plt.show()


In [ ]:
combined_adata

In [ ]:
filtered_combined_adata

### Filter: Min and max gene count per cell filtering

In [ ]:
# Visualize the number of genes per cell
sc.pl.violin(filtered_combined_adata, keys=['n_genes_by_counts'], jitter=True, log=False)

import seaborn as sns
sns.histplot(filtered_combined_adata.obs['n_genes_by_counts'], bins=50)

In [ ]:
# Set thresholds based on the previous scatter plots, removing low-quality cells and potential doublets.
mincount = 1900;
maxcount = 9000;

# Optional 
#mincount = combined_adata.obs['n_genes_by_counts'].quantile(0.01)
#maxcount = combined_adata.obs['n_genes_by_counts'].quantile(0.99)

In [ ]:
#sc.pp.filter_cells(combined_adata, min_counts=1000)
filtered_combined_adata = filtered_combined_adata[filtered_combined_adata.obs.n_genes_by_counts > mincount, :]
filtered_combined_adata = filtered_combined_adata[filtered_combined_adata.obs.n_genes_by_counts < maxcount, :]

In [ ]:
sc.pl.violin(filtered_combined_adata, keys=['n_genes_by_counts'], jitter=True, log=False)
sns.histplot(filtered_combined_adata.obs['n_genes_by_counts'], bins=50)

In [ ]:
filtered_combined_adata

### Filter: Min and max total transcript count per cell filtering

In [ ]:
# filtering based on total_counts is also common in single-cell RNA sequencing (scRNA-seq) quality control, and it's analogous to filtering based on n_genes_by_counts.
# Why Apply Min and Max Cutoffs on total_counts?
## Low-quality cells: Cells with very low total_counts can indicate cells with poor-quality RNA, dying cells, or cells with limited RNA content. They might also represent empty droplets or ambient RNA in droplet-based technologies like 10x Genomics.
## Doublets or Multiplets: An abnormally high total_counts might indicate doublets or multiplets, where two or more cells got captured together.
## Standardizing Sequencing Depth: While downstream normalization methods often account for differences in sequencing depth, extreme outliers can still introduce biases.

In [ ]:
# Histogram or Violin Plot: A visual inspection can help identify outliers or bimodal distributions.
sc.pl.violin(filtered_combined_adata, keys=['total_counts'], jitter=True, log=False)
    
import seaborn as sns
sns.histplot(filtered_combined_adata.obs['total_counts'], bins=50)

In [ ]:
mincount = 0; 
maxcount = 70000; 
#alternative
#mincount = filtered_combined_adata.obs['total_counts'].quantile(0.01)
#maxcount = filtered_combined_adata.obs['total_counts'].quantile(0.99)

In [ ]:
#sc.pp.filter_cells(combined_adata, min_counts=1000)
filtered_combined_adata = filtered_combined_adata[filtered_combined_adata.obs.total_counts > mincount, :]
filtered_combined_adata = filtered_combined_adata[filtered_combined_adata.obs.total_counts < maxcount, :]

In [ ]:
sns.histplot(filtered_combined_adata.obs['total_counts'], bins=50)

In [ ]:
filtered_combined_adata.obs

In [ ]:
filtered_combined_adata

### Filter: Doublet cell

In [ ]:
sc.pp.filter_genes(filtered_combined_adata, min_cells=3)

# Entferne doppelte Zellen mit Scrublet
scrub = scr.Scrublet(filtered_combined_adata.X)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

# Füge die Scrublet-Ergebnisse zum AnnData-Objekt hinzu
filtered_combined_adata.obs['doublet_scores'] = doublet_scores
filtered_combined_adata.obs['predicted_doublets'] = predicted_doublets

# Entferne vorhergesagte doppelte Zellen
filtered_combined_adata = filtered_combined_adata[~filtered_combined_adata.obs['predicted_doublets']]


In [ ]:
filtered_combined_adata

### Filter outlier cells

In [ ]:
# Define the function to check outliers
def is_outlier(filtered_combined_adata, metric: str, nmads: int):
    M = filtered_combined_adata.obs[metric]
    outlier = (M < np.median(M) - nmads * np.median(np.abs(M - np.median(M)))) | (
        np.median(M) + nmads * np.median(np.abs(M - np.median(M))) < M
    )
    return outlier


# Apply the outlier function to create a new column 'outlier' in adata.obs
filtered_combined_adata.obs["outlier"] = (
    is_outlier(filtered_combined_adata, "log1p_total_counts", 5)
    | is_outlier(filtered_combined_adata, "log1p_n_genes_by_counts", 5)
)

# Count the number of outliers
outlier_counts = filtered_combined_adata.obs.outlier.value_counts()
print(outlier_counts)


In [ ]:
# Filterung basierend auf Ausreißern in outlier und mt_outlier Spalten
outlier_filter = ~(filtered_combined_adata.obs["outlier"] )
adata_filtered = filtered_combined_adata[outlier_filter].copy()

# Überprüfen der Anzahl der verbleibenden Zellen nach der Filterung
print(f"Number of cells after filtering of low quality cells: {adata_filtered.n_obs}")

In [ ]:
adata_filtered

In [ ]:
sc.pl.violin(
    filtered_combined_adata,
    keys=['pct_counts_mito', 'pct_counts_ribo'],
    groupby='condition',
    jitter=0.4,
    rotation=45,
    stripplot=True,
    multi_panel=True,
    save='violin_mito_ribo.png'  # Save the plot as an image
)
plt.show()

### Normalization

In [ ]:
# You can use the median or mean total count across all cells as the target_sum. This ensures that the normalization doesn't excessively scale up very low-count cells or scale down very high-count cells.
#mean_counts = adata_filtereda.obs['total_counts'].mean()
median_counts = adata_filtered.obs['total_counts'].median()
median_counts

In [ ]:
sc.pp.normalize_total(adata_filtered, target_sum=1e4)
#sc.pp.normalize_total(adata_filtered, target_sum=median_counts)

In [ ]:
adata_filtered

### Log transformation

In [ ]:
# Check for large integers (e.g., > 50) in the data matrix
large_integers = (adata_filtered.X > 50).sum()
if large_integers > 0:
    print("please logtransform again.")
else:
    print("Data seems to be already log-transformed. No transformation applied.")

In [ ]:
sc.pp.log1p(adata_filtered)

In [ ]:
adata_filtered

### Dimension Reduction

In [ ]:
# Calculate PCA
sc.tl.pca(adata_filtered)

# Compute neighbors
sc.pp.neighbors(adata_filtered)

# Calculate clustering
sc.tl.leiden(adata_filtered)  # or sc.tl.louvain(adata)

# Compute UMAP (Optional)
sc.tl.umap(adata_filtered)

In [ ]:
sc.pl.pca(adata_filtered, color='condition')

In [ ]:
sc.pl.pca_variance_ratio(adata_filtered, log=True)

#### Save filtered and normalized file

In [ ]:
# Save adata --> here ID ensg number_for pseudotime
adata_filtered.write('/storage/users/data/PANC/H5AD_file/adata_filtered.h5ad')

In [ ]:
adata_filtered = sc.read('/storage/users/data/PANC/H5AD_file/adata_filtered.h5ad') 

## Remove 2D condition 

In [ ]:
print(adata_filtered.shape)

In [ ]:
# Apply filtering
adata_filtered_no2D = adata_filtered[adata_filtered.obs['condition'] != 'CTRL_2D'].copy()

In [ ]:
adata_filtered.obs['condition']

In [ ]:
print(adata_filtered_no2D.shape)

### Remove zero columns

In [ ]:
import numpy as np
from scipy.sparse import issparse

# Check if the matrix is sparse
if issparse(adata_filtered_no2D.X):
    zero_count = adata_filtered_no2D.X.size - adata_filtered_no2D.X.nnz
else:
    zero_count = np.count_nonzero(adata_filtered_no2D.X == 0)

total_elements = adata_filtered_no2D.X.shape[0] * adata_filtered_no2D.X.shape[1]
print(f"Total number of zeros: {zero_count}")
print(f"Percentage of zeros: {zero_count / total_elements * 100:.2f}%")

# Check for entirely zero rows or columns
if issparse(adata_filtered_no2D.X):
    zero_rows = np.squeeze(np.array((adata_filtered_no2D.X.sum(axis=1) == 0)))
    zero_cols = np.squeeze(np.array((adata_filtered_no2D.X.sum(axis=0) == 0)))
else:
    zero_rows = np.all(adata_filtered_no2D.X == 0, axis=1)
    zero_cols = np.all(adata_filtered_no2D.X == 0, axis=0)

print("Number of completely zero rows: ", np.sum(zero_rows))
print("Number of completely zero columns: ", np.sum(zero_cols))

In [ ]:
# Optionally, filter out zero rows and columns
if np.sum(zero_rows) > 0:
    adata_filtered_no2D = adata_filtered_no2D[~zero_rows, :]
if np.sum(zero_cols) > 0:
    adata_filtered_no2D = adata_filtered_no2D[:, ~zero_cols]

In [ ]:
print(adata_filtered.shape)

In [ ]:
print(adata_filtered_no2D.shape)

In [ ]:
print(adata_filtered.obs['condition'].unique())  # Before filtering
print(adata_filtered_no2D.obs['condition'].unique())  # After filtering

### Recomputing Metrics and Dimensionality Reduction After Data Filtering in Scanpy

In [ ]:
# Calculate PCA
sc.tl.pca(adata_filtered_no2D)

# Compute neighbors
sc.pp.neighbors(adata_filtered_no2D)

# Calculate clustering
sc.tl.leiden(adata_filtered_no2D)  # or sc.tl.louvain(adata)

# Compute UMAP (Optional)
sc.tl.umap(adata_filtered_no2D)


#### Save the NO2D data

In [ ]:
# Save adata --> here ID ensg number_for pseudotime
adata_filtered_no2D.write('/storage/users/data/PANC/H5AD_file/adata_filtered_no2D.h5ad')

In [ ]:
adata_filtered_no2D = sc.read('/storage/users/data/PANC/H5AD_file/adata_filtered_no2D.h5ad') 

## Highly Variable Gene Selection on adata_filtered_no2D

In [ ]:
import matplotlib.pyplot as plt

# Compute highly variable genes
sc.pp.highly_variable_genes(
    adata_filtered_no2D,
    n_top_genes=2000,
    min_mean=-2e-5,
    max_mean=2e-5,
    min_disp=0.5,
    max_disp=5
)

# Extract information for plotting
means = adata_filtered_no2D.var['means']
dispersions = adata_filtered_no2D.var['dispersions']
highly_variable = adata_filtered_no2D.var['highly_variable']

# Plot mean vs dispersion
plt.figure(figsize=(10, 6))
plt.scatter(means, dispersions, c='gray', s=10, alpha=0.5, label="All genes")
plt.scatter(
    means[highly_variable],
    dispersions[highly_variable],
    c='red',
    s=10,
    alpha=0.8,
    label="Highly Variable Genes"
)
plt.axhline(y=0.5, color='blue', linestyle='--', label='Min Dispersion')
plt.axhline(y=5, color='green', linestyle='--', label='Max Dispersion')
plt.axvline(x=-2e-5, color='purple', linestyle='--', label='Min Mean')
plt.axvline(x=2e-5, color='orange', linestyle='--', label='Max Mean')

plt.xlabel("Mean Expression")
plt.ylabel("Dispersion")
plt.title("Highly Variable Genes")
plt.legend()
plt.show()


In [ ]:
# Calculate means and inspect extreme values
gene_means = np.mean(adata_filtered_no2D.X, axis=0).A1 if issparse(adata_filtered_no2D.X) else np.mean(adata_filtered_no2D.X, axis=0)
print("Max mean:", np.max(gene_means))
print("Min mean:", np.min(gene_means))

In [ ]:
#sc.pp.highly_variable_genes(adata_filtered_no2D,n_top_genes=4000)

In [ ]:
# Identifying and focusing on highly variable genes improves the sensitivity of detecting cell populations.
sc.pp.highly_variable_genes(
    adata_filtered_no2D,
    n_top_genes=2000,
    min_mean=2e-5,  # Slightly below the min mean
    max_mean=5.3,   # Slightly above the max mean
    min_disp=0.5,    # Default dispersion threshold
    max_disp=5       # Default dispersion threshold
)

In [ ]:
# Determine a highly variable gene adata object
adata_filtered_no2D_hvg = adata_filtered_no2D[:, adata_filtered_no2D.var.highly_variable]

In [ ]:
adata_filtered_no2D_hvg

### Remove zero columns

In [ ]:
import numpy as np
from scipy.sparse import issparse

# Check if the matrix is sparse
if issparse(adata_filtered_no2D_hvg.X):
    zero_count = adata_filtered_no2D_hvg.X.size - adata_filtered_no2D_hvg.X.nnz
else:
    zero_count = np.count_nonzero(adata_filtered_no2D_hvg.X == 0)

total_elements = adata_filtered_no2D_hvg.X.shape[0] * adata_filtered_no2D_hvg.X.shape[1]
print(f"Total number of zeros: {zero_count}")
print(f"Percentage of zeros: {zero_count / total_elements * 100:.2f}%")

# Check for entirely zero rows or columns
if issparse(adata_filtered_no2D_hvg.X):
    zero_rows = np.squeeze(np.array((adata_filtered_no2D_hvg.X.sum(axis=1) == 0)))
    zero_cols = np.squeeze(np.array((adata_filtered_no2D_hvg.X.sum(axis=0) == 0)))
else:
    zero_rows = np.all(adata_filtered_no2D_hvg.X == 0, axis=1)
    zero_cols = np.all(adata_filtered_no2D_hvg.X == 0, axis=0)

print("Number of completely zero rows: ", np.sum(zero_rows))
print("Number of completely zero columns: ", np.sum(zero_cols))

In [ ]:
# Optionally, filter out zero rows and columns
if np.sum(zero_rows) > 0:
    adata_filtered_no2D_hvg = adata_filtered_no2D_hvg[~zero_rows, :]
if np.sum(zero_cols) > 0:
    adata_filtered_no2D_hvg = adata_filtered_no2D_hvg[:, ~zero_cols]

In [ ]:
print(adata_filtered.shape)

In [ ]:
print(adata_filtered_no2D.shape)

In [ ]:
print(adata_filtered_no2D_hvg.shape)

### Recomputing Metrics and Dimensionality Reduction After Data Filtering in Scanpy

In [ ]:
# Calculate PCA
sc.tl.pca(adata_filtered_no2D_hvg)

# Compute neighbors
sc.pp.neighbors(adata_filtered_no2D_hvg)

# Calculate clustering
sc.tl.leiden(adata_filtered_no2D_hvg)  # or sc.tl.louvain(adata)

# Compute UMAP (Optional)
sc.tl.umap(adata_filtered_no2D_hvg)


In [ ]:
sc.pl.pca(adata_filtered_no2D_hvg, color='condition')

#### Save the NO2D and HVG data

In [ ]:
adata_filtered_no2D_hvg.write('/storage/users/data/PANC/H5AD_file/adata_filtered_no2D_hvg.h5ad')

In [ ]:
adata_filtered_no2D_hvg = sc.read('/storage/users/data/PANC/H5AD_file/adata_filtered_no2D_hvg.h5ad') 

## Highly Variable Gene Selection on adata_filtered

In [ ]:
adata_filtered

In [ ]:
# Calculate means and inspect extreme values
gene_means = np.mean(adata_filtered.X, axis=0).A1 if issparse(adata_filtered.X) else np.mean(adata_filtered.X, axis=0)
print("Max mean:", np.max(gene_means))
print("Min mean:", np.min(gene_means))

In [ ]:
# Identifying and focusing on highly variable genes improves the sensitivity of detecting cell populations.
sc.pp.highly_variable_genes(
    adata_filtered,
    n_top_genes=4000,
    min_mean=3e-05,  # Slightly below the min mean
    max_mean=5.3,   # Slightly above the max mean
    min_disp=0.5,    # Default dispersion threshold
    max_disp=5       # Default dispersion threshold
)

In [ ]:
# Determine a highly variable gene adata object
adata_filtered_hvg = adata_filtered[:, adata_filtered.var.highly_variable]

In [ ]:
adata_filtered_hvg

### Remove zero columns

In [ ]:
import numpy as np
from scipy.sparse import issparse

# Check if the matrix is sparse
if issparse(adata_filtered_hvg.X):
    zero_count = adata_filtered_hvg.X.size - adata_filtered_hvg.X.nnz
else:
    zero_count = np.count_nonzero(adata_filtered_hvg.X == 0)

total_elements = adata_filtered_hvg.X.shape[0] * adata_filtered_hvg.X.shape[1]
print(f"Total number of zeros: {zero_count}")
print(f"Percentage of zeros: {zero_count / total_elements * 100:.2f}%")

# Check for entirely zero rows or columns
if issparse(adata_filtered_no2D_hvg.X):
    zero_rows = np.squeeze(np.array((adata_filtered_no2D_hvg.X.sum(axis=1) == 0)))
    zero_cols = np.squeeze(np.array((adata_filtered_no2D_hvg.X.sum(axis=0) == 0)))
else:
    zero_rows = np.all(adata_filtered_no2D_hvg.X == 0, axis=1)
    zero_cols = np.all(adata_filtered_no2D_hvg.X == 0, axis=0)

print("Number of completely zero rows: ", np.sum(zero_rows))
print("Number of completely zero columns: ", np.sum(zero_cols))

In [ ]:
# Optionally, filter out zero rows and columns
if np.sum(zero_rows) > 0:
    adata_filtered_no2D_hvg = adata_filtered_no2D_hvg[~zero_rows, :]
if np.sum(zero_cols) > 0:
    adata_filtered_no2D_hvg = adata_filtered_no2D_hvg[:, ~zero_cols]

In [ ]:
print(adata_filtered.shape)

In [ ]:
print(adata_filtered_no2D.shape)

In [ ]:
print(adata_filtered_hvg.shape)

In [ ]:
print(adata_filtered_no2D_hvg.shape)

### Recomputing Metrics and Dimensionality Reduction After Data Filtering in Scanpy

In [ ]:
# Calculate PCA
sc.tl.pca(adata_filtered_hvg)

# Compute neighbors
sc.pp.neighbors(adata_filtered_hvg)

# Calculate clustering
sc.tl.leiden(adata_filtered_hvg)  # or sc.tl.louvain(adata)

# Compute UMAP (Optional)
sc.tl.umap(adata_filtered_hvg)


In [ ]:
#sc.tl.pca(combined_adata)
sc.tl.pca(adata_filtered_hvg, use_highly_variable=False)
sc.pl.pca(adata_filtered_hvg, color='condition')

#### Save the HVG data

In [ ]:
adata_filtered_hvg.write('/storage/users/data/PANC/H5AD_file/adata_filtered_hvg.h5ad')

In [ ]:
adata_filtered_hvg = sc.read('/storage/users/data/PANC/H5AD_file/adata_filtered_hvg.h5ad') 

### Filter: CMO tags

In [ ]:
# Identify CMO tags (or other spike-ins)
combined_adata.var["CMO"] = combined_adata.var_names.str.contains("CMO")

In [ ]:
# Get Percentage of CMOs in cells
# Filter the Data for CMO Tags
cmo_data = combined_adata[:, combined_adata.var["CMO"].values].X
# Calculate the Percentage:
cmo_counts_per_cell = cmo_data.sum(axis=1)
total_counts_per_cell = combined_adata.X.sum(axis=1)
pct_cmo_per_cell = (cmo_counts_per_cell / total_counts_per_cell) * 100
# Add to the obs DataFrame:
combined_adata.obs["pct_cmo"] = pct_cmo_per_cell
print(combined_adata.obs["pct_cmo"])

#### Remove CMO tags

In [ ]:
combined_adata.var["CMO"]

In [ ]:
combined_adata = combined_adata[:, ~combined_adata.var["CMO"].values]

In [ ]:
combined_adata.var["CMO"]

In [ ]:
combined_adata

### Calculate several QC metrics that will be stored in the adata Object

In [ ]:
sc.pp.calculate_qc_metrics(combined_adata, inplace=True)
print(combined_adata)

### Filter: Mitochondrial and Ribosomal Genes

#### Create dictionary ensembl2symbol

In [ ]:
import mygene

# Extract Ensembl Gene IDs from combined_adata
ensembl_ids = combined_adata.var_names.tolist()

# Initialize MyGene.info client
mg = mygene.MyGeneInfo()

# Query MyGene.info for mappings
print("Querying MyGene.info...")
results = mg.querymany(
    ensembl_ids,
    scopes="ensembl.gene",
    fields="symbol",
    species="human"
)

# Initialize an empty dictionary to store the mapping
ensembl_to_gene_name = {}

# Populate the dictionary
print("Processing results...")
for res in results:
    ensembl_id = res.get("query")
    if "notfound" in res:
        # Skip if not found
        continue
    # Use the symbol field as the gene name
    ensembl_to_gene_name[ensembl_id] = res.get("symbol", ensembl_id)

# Display the resulting dictionary
print("\nEnsembl to Gene Name mapping:")
#print(ensembl_to_gene_name)


In [ ]:
import mygene

# Extract Ensembl Gene IDs from combined_adata
ensembl_ids = combined_adata.var_names.tolist()

# Initialize MyGene.info client
mg = mygene.MyGeneInfo()

# Query MyGene.info for mappings
print("Querying MyGene.info...")
results = mg.querymany(
    ensembl_ids,
    scopes="ensembl.gene",
    fields="symbol",
    species="human"
)

# Initialize an empty dictionary to store the mapping
ensembl_to_gene_name = {}

# Populate the dictionary
print("Processing results...")
for res in results:
    ensembl_id = res.get("query")
    if "notfound" in res:
        # Skip if not found
        continue
    # Use the symbol field as the gene name
    ensembl_to_gene_name[ensembl_id] = res.get("symbol", ensembl_id)

# Display the resulting dictionary
print("\nEnsembl to Gene Name mapping:")
print(ensembl_to_gene_name)


#### Add gene symbol to combined_adata object

##### Annotate Gene Symbol

In [ ]:
# Assuming ensembl_to_gene_name is a dictionary where the keys are Ensembl IDs and the values are gene symbols
combined_adata.var["gene_symbol"] = combined_adata.var_names.map(ensembl_to_gene_name)

In [ ]:
combined_adata.var["gene_symbol"]

##### Fill empty Symbols with Ensembl IDs

In [ ]:
combined_adata.var["ensembl_gene_id"] = combined_adata.var_names.astype(str)
combined_adata.var["gene_symbol"].fillna(combined_adata.var["ensembl_gene_id"], inplace=True)
print("Remaining NaN in 'gene_symbol':", combined_adata.var["gene_symbol"].isna().sum())


In [ ]:
combined_adata.var_names

In [ ]:
combined_adata.var["gene_symbol"]

In [ ]:
combined_adata

#### Analyze Mito and Ribo Genes

In [ ]:
# After loading your data, identify mitochondrial and ribosomal genes
#combined_adata.var["mito"] = combined_adata.var_names.str.startswith("MT-") | combined_adata.var_names.str.startswith("mt-")
#combined_adata.var["ribo"] = combined_adata.var_names.str.startswith("RPS") | combined_adata.var_names.str.startswith("RPL")

# Using the gene_symbol column to identify mitochondrial and ribosomal genes
combined_adata.var["mito"] = combined_adata.var["gene_symbol"].str.startswith("MT-") | combined_adata.var["gene_symbol"].str.startswith("mt-")
combined_adata.var["ribo"] = combined_adata.var["gene_symbol"].str.startswith("RPS") | combined_adata.var["gene_symbol"].str.startswith("RPL")


#### Compute QC metrics, considering both mitochondrial and ribosomal genes

In [ ]:
# Compute QC metrics, considering both mitochondrial and ribosomal genes
sc.pp.calculate_qc_metrics(combined_adata, qc_vars=["mito", "ribo"], percent_top=None, inplace=True)

In [ ]:
# Get a list of the newly annotated adata object and the Mitochondrial count
print(combined_adata)
combined_adata.obs.pct_counts_mito


In [ ]:
# Sort the cells by Percentage Mitochondrium and order by highest percentage first
sorted_adata = combined_adata[combined_adata.obs['pct_counts_mito'].sort_values(ascending=False).index]
sorted_adata.obs.pct_counts_mito

#### Visualize cell's mitchondrial and ribosomal gene expression

In [ ]:
# Visualize basic QC metrics, including ribosomal content:
sc.pl.violin(combined_adata, ['n_genes_by_counts', 'total_counts'])
sc.pl.violin(combined_adata, ['pct_counts_mito','pct_counts_ribo'])
import scanpy as sc

sc.pl.violin(
    combined_adata,
    keys=['pct_counts_mito', 'pct_counts_ribo'],
    groupby='condition',
    jitter=0.4,
    rotation=45,
    stripplot=True,
    multi_panel=True,
    save='violin_mito_ribo.png'  # Save the plot as an image
)
plt.show()

sc.pl.scatter(combined_adata, x='total_counts', y='pct_counts_mito')
sc.pl.scatter(combined_adata, x='total_counts', y='pct_counts_ribo')
sc.pl.scatter(combined_adata, x='total_counts', y='n_genes_by_counts')



#### Filter Mito and Ribo according to a certain percentage (e.g. 10 and 50%)

##### Delete faulty ribo/mito: Old logic to filter over all conditions

In [ ]:
combined_adata.obs['condition']

##### Not used!!: Delete faulty ribo/mito: new logic to filter conditions wise

In [ ]:
import numpy as np
import scanpy as sc

# Recalculate QC metrics if needed
sc.pp.calculate_qc_metrics(combined_adata, qc_vars=["mito", "ribo"], percent_top=None, inplace=True)

# Initialize lists to store percentile thresholds for each condition
mito_thresholds = {}
ribo_thresholds = {}

# Calculate percentiles per condition
for condition in combined_adata.obs['condition'].unique():
    # Subset data for the current condition
    condition_data = combined_adata.obs[combined_adata.obs['condition'] == condition]

    if condition_data.shape[0] == 0:
        print(f"Skipping condition: {condition} (no cells)")
        continue

    # Calculate percentiles
    mito_upper = np.percentile(condition_data['pct_counts_mito'], 90)
    #ribo_lower = np.percentile(condition_data['pct_counts_ribo'], 5)
    ribo_upper = np.percentile(condition_data['pct_counts_ribo'], 90)

    # Store thresholds for reference
    mito_thresholds[condition] = mito_upper
    ribo_thresholds[condition] = (ribo_lower, ribo_upper)

    print(f"{condition}: mito_upper={mito_upper:.2f}, ribo_lower={ribo_lower:.2f}, ribo_upper={ribo_upper:.2f}")

# Apply filtering
filtered_adata = combined_adata[
    combined_adata.obs.apply(
        lambda x: (
            x['pct_counts_mito'] <= mito_thresholds[x['condition']] and
            ribo_thresholds[x['condition']][0] <= x['pct_counts_ribo'] <= ribo_thresholds[x['condition']][1]
        ),
        axis=1
    ),
    :
]

print(f"Filtered data shape: {filtered_adata.shape}")

In [ ]:
# Filter cells based on mitochondrial and ribosomal content:
# Cells with high mitochondrial gene expression might be undergoing apoptosis, so you might want to exclude them.
sc.pp.calculate_qc_metrics(filtered_adata, qc_vars=["mito", "ribo"], percent_top=None, inplace=True)
filtered_adata = filtered_adata[filtered_adata.obs.pct_counts_mito < 10, :]
# high levels of ribosomal RNA (rRNA) can indicate incomplete poly-A tail capture or contamination. Thus, by filtering out cells with excessive ribosomal transcripts, we're likely removing lower-quality cells.
filtered_adata = filtered_adata[filtered_adata.obs.pct_counts_ribo < 45, :]

#### Redefine object according to which mito and ribo filter method used

In [ ]:
filtered_combined_adata = filtered_adata.copy()
#filtered_combined_adata = combined_adata.copy()

#### View and evaluate filtering

In [ ]:
sc.pp.calculate_qc_metrics(filtered_combined_adata, qc_vars=["mito", "ribo"], percent_top=None, inplace=True)
sc.pl.violin(
    filtered_combined_adata,
    keys=['pct_counts_mito', 'pct_counts_ribo'],
    groupby='condition',
    jitter=0.4,
    rotation=45,
    stripplot=True,
    multi_panel=True,
    save='violin_mito_ribo.png'  # Save the plot as an image
)
plt.show()


In [ ]:
combined_adata

In [ ]:
filtered_combined_adata

### Filter: Min and max gene count per cell filtering

In [ ]:
# Visualize the number of genes per cell
sc.pl.violin(filtered_combined_adata, keys=['n_genes_by_counts'], jitter=True, log=False)

import seaborn as sns
sns.histplot(filtered_combined_adata.obs['n_genes_by_counts'], bins=50)

In [ ]:
# Set thresholds based on the previous scatter plots, removing low-quality cells and potential doublets.
mincount = 1900;
maxcount = 9000;

# Optional 
#mincount = combined_adata.obs['n_genes_by_counts'].quantile(0.01)
#maxcount = combined_adata.obs['n_genes_by_counts'].quantile(0.99)

In [ ]:
#sc.pp.filter_cells(combined_adata, min_counts=1000)
filtered_combined_adata = filtered_combined_adata[filtered_combined_adata.obs.n_genes_by_counts > mincount, :]
filtered_combined_adata = filtered_combined_adata[filtered_combined_adata.obs.n_genes_by_counts < maxcount, :]

In [ ]:
sc.pl.violin(filtered_combined_adata, keys=['n_genes_by_counts'], jitter=True, log=False)
sns.histplot(filtered_combined_adata.obs['n_genes_by_counts'], bins=50)

In [ ]:
filtered_combined_adata

### Filter: Min and max total transcript count per cell filtering

In [ ]:
# filtering based on total_counts is also common in single-cell RNA sequencing (scRNA-seq) quality control, and it's analogous to filtering based on n_genes_by_counts.
# Why Apply Min and Max Cutoffs on total_counts?
## Low-quality cells: Cells with very low total_counts can indicate cells with poor-quality RNA, dying cells, or cells with limited RNA content. They might also represent empty droplets or ambient RNA in droplet-based technologies like 10x Genomics.
## Doublets or Multiplets: An abnormally high total_counts might indicate doublets or multiplets, where two or more cells got captured together.
## Standardizing Sequencing Depth: While downstream normalization methods often account for differences in sequencing depth, extreme outliers can still introduce biases.

In [ ]:
# Histogram or Violin Plot: A visual inspection can help identify outliers or bimodal distributions.
sc.pl.violin(filtered_combined_adata, keys=['total_counts'], jitter=True, log=False)
    
import seaborn as sns
sns.histplot(filtered_combined_adata.obs['total_counts'], bins=50)

In [ ]:
mincount = 0; 
maxcount = 70000; 
#alternative
#mincount = filtered_combined_adata.obs['total_counts'].quantile(0.01)
#maxcount = filtered_combined_adata.obs['total_counts'].quantile(0.99)

In [ ]:
#sc.pp.filter_cells(combined_adata, min_counts=1000)
filtered_combined_adata = filtered_combined_adata[filtered_combined_adata.obs.total_counts > mincount, :]
filtered_combined_adata = filtered_combined_adata[filtered_combined_adata.obs.total_counts < maxcount, :]

In [ ]:
sns.histplot(filtered_combined_adata.obs['total_counts'], bins=50)

In [ ]:
filtered_combined_adata.obs

In [ ]:
filtered_combined_adata

### Filter: Doublet cell

In [ ]:
sc.pp.filter_genes(filtered_combined_adata, min_cells=3)

# Entferne doppelte Zellen mit Scrublet
scrub = scr.Scrublet(filtered_combined_adata.X)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

# Füge die Scrublet-Ergebnisse zum AnnData-Objekt hinzu
filtered_combined_adata.obs['doublet_scores'] = doublet_scores
filtered_combined_adata.obs['predicted_doublets'] = predicted_doublets

# Entferne vorhergesagte doppelte Zellen
filtered_combined_adata = filtered_combined_adata[~filtered_combined_adata.obs['predicted_doublets']]


In [ ]:
filtered_combined_adata

### Filter outlier cells

In [ ]:
# Define the function to check outliers
def is_outlier(filtered_combined_adata, metric: str, nmads: int):
    M = filtered_combined_adata.obs[metric]
    outlier = (M < np.median(M) - nmads * np.median(np.abs(M - np.median(M)))) | (
        np.median(M) + nmads * np.median(np.abs(M - np.median(M))) < M
    )
    return outlier


# Apply the outlier function to create a new column 'outlier' in adata.obs
filtered_combined_adata.obs["outlier"] = (
    is_outlier(filtered_combined_adata, "log1p_total_counts", 5)
    | is_outlier(filtered_combined_adata, "log1p_n_genes_by_counts", 5)
)

# Count the number of outliers
outlier_counts = filtered_combined_adata.obs.outlier.value_counts()
print(outlier_counts)


In [ ]:
# Filterung basierend auf Ausreißern in outlier und mt_outlier Spalten
outlier_filter = ~(filtered_combined_adata.obs["outlier"] )
adata_filtered = filtered_combined_adata[outlier_filter].copy()

# Überprüfen der Anzahl der verbleibenden Zellen nach der Filterung
print(f"Number of cells after filtering of low quality cells: {adata_filtered.n_obs}")

In [ ]:
adata_filtered

In [ ]:
sc.pl.violin(
    filtered_combined_adata,
    keys=['pct_counts_mito', 'pct_counts_ribo'],
    groupby='condition',
    jitter=0.4,
    rotation=45,
    stripplot=True,
    multi_panel=True,
    save='violin_mito_ribo.png'  # Save the plot as an image
)
plt.show()

### Normalization

In [ ]:
# You can use the median or mean total count across all cells as the target_sum. This ensures that the normalization doesn't excessively scale up very low-count cells or scale down very high-count cells.
#mean_counts = adata_filtereda.obs['total_counts'].mean()
median_counts = adata_filtered.obs['total_counts'].median()
median_counts

In [ ]:
sc.pp.normalize_total(adata_filtered, target_sum=1e4)
#sc.pp.normalize_total(adata_filtered, target_sum=median_counts)

In [ ]:
adata_filtered

### Log transformation

In [ ]:
# Check for large integers (e.g., > 50) in the data matrix
large_integers = (adata_filtered.X > 50).sum()
if large_integers > 0:
    print("please logtransform again.")
else:
    print("Data seems to be already log-transformed. No transformation applied.")

In [ ]:
sc.pp.log1p(adata_filtered)

In [ ]:
adata_filtered

### Dimension Reduction

In [ ]:
# Calculate PCA
sc.tl.pca(adata_filtered)

# Compute neighbors
sc.pp.neighbors(adata_filtered)

# Calculate clustering
sc.tl.leiden(adata_filtered)  # or sc.tl.louvain(adata)

# Compute UMAP (Optional)
sc.tl.umap(adata_filtered)

In [ ]:
sc.pl.pca(adata_filtered, color='condition')

In [ ]:
sc.pl.pca_variance_ratio(adata_filtered, log=True)

### Save filtered and normalized file

In [ ]:
# Save adata --> here ID ensg number_for pseudotime
adata_filtered.write('/storage/users/data/PANC/H5AD_file/adata_filtered.h5ad')

In [ ]:
adata_filtered = sc.read('/storage/users/data/PANC/H5AD_file/adata_filtered.h5ad') 

In [ ]:
print(adata_filtered.shape)

In [ ]:
# Apply filtering
adata_filtered_no2D = adata_filtered[adata_filtered.obs['condition'] != 'CTRL_2D'].copy()

In [ ]:
adata_filtered.obs['condition']

In [ ]:
print(adata_filtered_no2D.shape)

### Remove zero columns

In [ ]:
import numpy as np
from scipy.sparse import issparse

# Check if the matrix is sparse
if issparse(adata_filtered_no2D.X):
    zero_count = adata_filtered_no2D.X.size - adata_filtered_no2D.X.nnz
else:
    zero_count = np.count_nonzero(adata_filtered_no2D.X == 0)

total_elements = adata_filtered_no2D.X.shape[0] * adata_filtered_no2D.X.shape[1]
print(f"Total number of zeros: {zero_count}")
print(f"Percentage of zeros: {zero_count / total_elements * 100:.2f}%")

# Check for entirely zero rows or columns
if issparse(adata_filtered_no2D.X):
    zero_rows = np.squeeze(np.array((adata_filtered_no2D.X.sum(axis=1) == 0)))
    zero_cols = np.squeeze(np.array((adata_filtered_no2D.X.sum(axis=0) == 0)))
else:
    zero_rows = np.all(adata_filtered_no2D.X == 0, axis=1)
    zero_cols = np.all(adata_filtered_no2D.X == 0, axis=0)

print("Number of completely zero rows: ", np.sum(zero_rows))
print("Number of completely zero columns: ", np.sum(zero_cols))

In [ ]:
# Optionally, filter out zero rows and columns
if np.sum(zero_rows) > 0:
    adata_filtered_no2D = adata_filtered_no2D[~zero_rows, :]
if np.sum(zero_cols) > 0:
    adata_filtered_no2D = adata_filtered_no2D[:, ~zero_cols]

In [ ]:
print(adata_filtered.shape)

In [ ]:
print(adata_filtered_no2D.shape)

In [ ]:
print(adata_filtered.obs['condition'].unique())  # Before filtering
print(adata_filtered_no2D.obs['condition'].unique())  # After filtering

### Recomputing Metrics and Dimensionality Reduction After Data Filtering in Scanpy

In [ ]:
# Calculate PCA
sc.tl.pca(adata_filtered_no2D)

# Compute neighbors
sc.pp.neighbors(adata_filtered_no2D)

# Calculate clustering
sc.tl.leiden(adata_filtered_no2D)  # or sc.tl.louvain(adata)

# Compute UMAP (Optional)
sc.tl.umap(adata_filtered_no2D)


### Save

In [ ]:
# Save adata --> here ID ensg number_for pseudotime
adata_filtered_no2D.write('/storage/users/data/PANC/H5AD_file/adata_filtered_no2D.h5ad')

In [ ]:
adata_filtered_no2D = sc.read('/storage/users/data/PANC/H5AD_file/adata_filtered_no2D.h5ad') 

In [ ]:
import matplotlib.pyplot as plt

# Compute highly variable genes
sc.pp.highly_variable_genes(
    adata_filtered_no2D,
    n_top_genes=2000,
    min_mean=-2e-5,
    max_mean=2e-5,
    min_disp=0.5,
    max_disp=5
)

# Extract information for plotting
means = adata_filtered_no2D.var['means']
dispersions = adata_filtered_no2D.var['dispersions']
highly_variable = adata_filtered_no2D.var['highly_variable']

# Plot mean vs dispersion
plt.figure(figsize=(10, 6))
plt.scatter(means, dispersions, c='gray', s=10, alpha=0.5, label="All genes")
plt.scatter(
    means[highly_variable],
    dispersions[highly_variable],
    c='red',
    s=10,
    alpha=0.8,
    label="Highly Variable Genes"
)
plt.axhline(y=0.5, color='blue', linestyle='--', label='Min Dispersion')
plt.axhline(y=5, color='green', linestyle='--', label='Max Dispersion')
plt.axvline(x=-2e-5, color='purple', linestyle='--', label='Min Mean')
plt.axvline(x=2e-5, color='orange', linestyle='--', label='Max Mean')

plt.xlabel("Mean Expression")
plt.ylabel("Dispersion")
plt.title("Highly Variable Genes")
plt.legend()
plt.show()


In [ ]:
# Calculate means and inspect extreme values
gene_means = np.mean(adata_filtered_no2D.X, axis=0).A1 if issparse(adata_filtered_no2D.X) else np.mean(adata_filtered_no2D.X, axis=0)
print("Max mean:", np.max(gene_means))
print("Min mean:", np.min(gene_means))

In [ ]:
#sc.pp.highly_variable_genes(adata_filtered_no2D,n_top_genes=4000)

In [ ]:
# Identifying and focusing on highly variable genes improves the sensitivity of detecting cell populations.
sc.pp.highly_variable_genes(
    adata_filtered_no2D,
    n_top_genes=2000,
    min_mean=2e-5,  # Slightly below the min mean
    max_mean=5.3,   # Slightly above the max mean
    min_disp=0.5,    # Default dispersion threshold
    max_disp=5       # Default dispersion threshold
)

In [ ]:
# Determine a highly variable gene adata object
adata_filtered_no2D_hvg = adata_filtered_no2D[:, adata_filtered_no2D.var.highly_variable]

In [ ]:
adata_filtered_no2D_hvg

### Remove zero columns

In [ ]:
import numpy as np
from scipy.sparse import issparse

# Check if the matrix is sparse
if issparse(adata_filtered_no2D_hvg.X):
    zero_count = adata_filtered_no2D_hvg.X.size - adata_filtered_no2D_hvg.X.nnz
else:
    zero_count = np.count_nonzero(adata_filtered_no2D_hvg.X == 0)

total_elements = adata_filtered_no2D_hvg.X.shape[0] * adata_filtered_no2D_hvg.X.shape[1]
print(f"Total number of zeros: {zero_count}")
print(f"Percentage of zeros: {zero_count / total_elements * 100:.2f}%")

# Check for entirely zero rows or columns
if issparse(adata_filtered_no2D_hvg.X):
    zero_rows = np.squeeze(np.array((adata_filtered_no2D_hvg.X.sum(axis=1) == 0)))
    zero_cols = np.squeeze(np.array((adata_filtered_no2D_hvg.X.sum(axis=0) == 0)))
else:
    zero_rows = np.all(adata_filtered_no2D_hvg.X == 0, axis=1)
    zero_cols = np.all(adata_filtered_no2D_hvg.X == 0, axis=0)

print("Number of completely zero rows: ", np.sum(zero_rows))
print("Number of completely zero columns: ", np.sum(zero_cols))

In [ ]:
# Optionally, filter out zero rows and columns
if np.sum(zero_rows) > 0:
    adata_filtered_no2D_hvg = adata_filtered_no2D_hvg[~zero_rows, :]
if np.sum(zero_cols) > 0:
    adata_filtered_no2D_hvg = adata_filtered_no2D_hvg[:, ~zero_cols]

In [ ]:
print(adata_filtered.shape)

In [ ]:
print(adata_filtered_no2D.shape)

In [ ]:
print(adata_filtered_no2D_hvg.shape)

### Recomputing Metrics and Dimensionality Reduction After Data Filtering in Scanpy

In [ ]:
# Calculate PCA
sc.tl.pca(adata_filtered_no2D_hvg)

# Compute neighbors
sc.pp.neighbors(adata_filtered_no2D_hvg)

# Calculate clustering
sc.tl.leiden(adata_filtered_no2D_hvg)  # or sc.tl.louvain(adata)

# Compute UMAP (Optional)
sc.tl.umap(adata_filtered_no2D_hvg)


In [ ]:
sc.pl.pca(adata_filtered_no2D_hvg, color='condition')

### Save the data

In [ ]:
adata_filtered_no2D_hvg.write('/storage/users/data/PANC/H5AD_file/adata_filtered_no2D_hvg.h5ad')

In [ ]:
adata_filtered_no2D_hvg = sc.read('/storage/users/data/PANC/H5AD_file/adata_filtered_no2D_hvg.h5ad') 